## Abaqus cantilever example

This notebook runs the built-in steel cantilever example directly with the project's Python tools. It does not use the AI assistant or Phoenix and does not make OpenAI API calls.

Run from the repository root with a licensed Abaqus installation and an `abaqus` command on PATH. Executing the cells launches real Abaqus processes, consumes compute and any applicable license resources, and writes inputs and results under `src/abaqus_files/`. Jupyter and its Python kernel must be installed separately if your environment does not already include them. The generated demo uses Pa for stress and the extraction tool converts it to MPa.


In [ ]:
import re
import uuid
from pathlib import Path

if not Path("src/tools.py").is_file():
    raise RuntimeError("Start the notebook from the repository root.")

from src.tools import generate_input_file, run_abaqus, extract_von_mises_stress_from_ODB

# Keep this notebook's latest verified result separate from other sessions.
session_id = uuid.uuid4().hex
input_name = None
run_succeeded = False
stress_mpa = None


## Generate, run, and extract

The following cells create the fixed cantilever model at a 0.01 m downward displacement. They use the generated filename explicitly and stop on errors instead of reading an earlier result. Re-run them in order.


In [ ]:
# Invalidate old state before generating another input.
input_name = None
run_succeeded = False
stress_mpa = None
if not globals().get("session_id"):
    raise RuntimeError("Run the setup cell first.")

generation = generate_input_file(0.01)
match = re.fullmatch(r"Input file generated: `([^`]+)` \(demo stress unit: Pa\)\.", generation)
if not match:
    raise RuntimeError(f"Input generation failed: {generation}")
input_name = match.group(1)
print(f"Generated input: {input_name}")


In [ ]:
# Always clear the previous result before starting a new run.
run_succeeded = False
stress_mpa = None
if not globals().get("session_id") or not globals().get("input_name"):
    raise RuntimeError("Run the setup and input-generation cells first.")

job_result = run_abaqus(inp_file_name=input_name, stress_unit="Pa", session_id=session_id)
if not job_result.startswith("Abaqus job completed successfully for `"):
    raise RuntimeError(f"Abaqus run failed: {job_result}")
run_succeeded = True
print(job_result)


In [ ]:
stress_mpa = None
if not globals().get("run_succeeded"):
    raise RuntimeError("Run the Abaqus job successfully before extracting stress.")

stress_result = extract_von_mises_stress_from_ODB(stress_unit="Pa", session_id=session_id)
match = re.fullmatch(r"von_mises stress is ([0-9]+(?:\.[0-9]+)?(?:[eE][+-]?[0-9]+)?) MPa", stress_result)
if not match:
    raise RuntimeError(f"Stress extraction failed: {stress_result}")
stress_mpa = float(match.group(1))
print(f"Maximum von Mises stress: {stress_mpa:g} MPa")


The job and output database are retained in a unique directory under `src/abaqus_files/runs/`. A successful run and output reading do not independently validate the physical model or its assumptions; inspect the model and results before drawing engineering conclusions.
